# PD-001 — one-click development

Import into the original minicells Kaggle notebook and click Run All. The existing GITHUB_TOKEN must be enabled; HF_TOKEN is optional for public training data. Assets, hashes, seeds and run identity are prepared automatically. Preflight failure stops the run. Training uses independent GPU workers, profiling runs afterward, and small result metadata are returned to GitHub. Repeating Run All reuses checkpoints when this output directory is retained. Confirmation is never included.


In [ ]:
from pathlib import Path
import subprocess, sys
MODE = 'auto'  # preflight -> registered experiment; any failure stops
SOURCE_COMMIT = '2e77369b0df1ce704863a815dacc48301736dde0'
ROOT = Path('/kaggle/working/minicells-pd001-' + SOURCE_COMMIT[:12])
if MODE not in ('auto','audit','params','dry-run','preflight'):
    raise RuntimeError('Choose auto or an engineering diagnostic mode')
if SOURCE_COMMIT.startswith('__'):
    raise RuntimeError('Import the committed notebook with its pinned source')
ROOT.mkdir(parents=True,exist_ok=True)
def git(*args):
    print('Setup: git ' + ' '.join(args),flush=True)
    subprocess.run(['git',*args],cwd=ROOT,check=True)
ready = False
if (ROOT/'.git').exists():
    head = subprocess.run(['git','rev-parse','HEAD'],cwd=ROOT,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE)
    ready = head.returncode == 0 and head.stdout.strip() == SOURCE_COMMIT and (ROOT/'research/projects/cellular-language-models/experiments/006-path-first-clm/extension-parameter-depth/run_pd001.py').exists()
if not ready:
    if not (ROOT/'.git').exists():
        git('init')
        git('remote','add','origin','https://github.com/ArcheLabs/mini-cells.git')
    origin = subprocess.check_output(['git','remote','get-url','origin'],cwd=ROOT,text=True).strip()
    if origin != 'https://github.com/ArcheLabs/mini-cells.git':
        raise RuntimeError('Unexpected source checkout origin')
    git('config','remote.origin.promisor','true')
    git('config','remote.origin.partialclonefilter','blob:none')
    git('sparse-checkout','init','--no-cone')
    git('sparse-checkout','set','--no-cone','/pyproject.toml','/src/','/shared/python/',
        '/research/projects/cellular-language-models/experiments/006-path-first-clm/',
        '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/',
        '/tests/research/path_first_pd001/',
        '/research/governance/formal_seed_registry.json',
        '/research/projects/**/config/*.json', '/research/projects/**/configs/*.json',
        '/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/',
        '/research/projects/continual-learning/context/04-continual-learning/scripts/research/')
    git('fetch','--depth=1','--filter=blob:none','origin',SOURCE_COMMIT)
    git('checkout','--detach',SOURCE_COMMIT)


In [ ]:
ENTRY = ROOT/'research/projects/cellular-language-models/experiments/006-path-first-clm/extension-parameter-depth/run_pd001.py'
if not ENTRY.exists():
    raise RuntimeError('Checkout is incomplete; preserve output, remove only the incomplete code checkout and retry')
print('Automatic preflight, assets, development training and return; failures stop the flow',flush=True)
import importlib.util
missing = [package for package,module in [('datasets','datasets'),('tokenizers','tokenizers'),('huggingface_hub','huggingface_hub')] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',*missing],check=True)
import os, time
out = Path('/kaggle/working/pd001-development-' + SOURCE_COMMIT[:12])
env = {**os.environ, 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1'}
if MODE != 'auto':
    env['CUDA_VISIBLE_DEVICES'] = ''
config = ENTRY.parent/'config/proposed.json'
command = [sys.executable,str(ENTRY),'--mode',MODE,'--device','cpu','--config',str(config),'--output',str(out)]
if MODE == 'auto':
    command += ['--devices','auto']
subprocess.run(command,cwd=ROOT,env=env,check=True)
print({'status':'AUTOMATIC_FLOW_COMPLETED','output':str(out),'scope':'development_only'},flush=True)
